In [ ]:
%load_ext watermark


In [ ]:
import os

import numpy as np
import pandas as pd
import seaborn as sns
from teeplot import teeplot as tp


In [ ]:
%watermark -diwmuv -iv


In [ ]:
teeplot_subdir = os.environ.get(
    "NOTEBOOK_NAME", "2026-03-23-d2h-memcpy-diagnostics"
)
teeplot_subdir


## Fetch Data


In [ ]:
url = "https://github.com/user-attachments/files/26186490/bookends.csv"
df = pd.read_csv(url)
print(f"{len(df)} mismatched bookend records")
df.head()


## Parse Bookends and Classify Parity Errors


In [ ]:
df["start_bookend"] = df["data_hex"].str[:8].apply(int, base=16)
df["end_bookend"] = df["data_hex"].str[-8:].apply(int, base=16)
df["diff"] = df["end_bookend"] - df["start_bookend"]

df["parity_error"] = np.where(
    df["diff"] == 512,
    "+1 parity",
    np.where(df["diff"] == -512, "-1 parity", "other"),
)
print(df["parity_error"].value_counts())
df.head()


## Spatial Coordinates


In [ ]:
nRow = 1170
nCol = 755
df["x"] = df["position"] // nCol  # row
df["y"] = df["position"] % nCol  # column

print(f"Row range: {df['x'].min()}--{df['x'].max()}")
print(f"Col range: {df['y'].min()}--{df['y'].max()}")


## Plot 1 --- Spatial Distribution


In [ ]:
with tp.teed(
    sns.scatterplot,
    data=df,
    x="y",
    y="x",
    s=1,
    alpha=0.4,
    color="tab:red",
    edgecolor="none",
    legend=False,
    teeplot_subdir=teeplot_subdir,
) as ax:
    ax.set_xlim(0, nCol)
    ax.set_ylim(nRow, 0)
    ax.set_aspect("equal")
    ax.set_xlabel("Column")
    ax.set_ylabel("Row")
    ax.set_title(
        f"Spatial Distribution of Bookend Mismatches (n={len(df):,})",
    )
    ax.figure.set_size_inches(1.5, 1.5)


## Plot 2 --- Temporal Distribution Across Layers


In [ ]:
df_layer = (
    df.groupby(["layer", "parity_error"]).size().reset_index(name="count")
)

with tp.teed(
    sns.histplot,
    data=df_layer,
    x="layer",
    hue="parity_error",
    weights="count",
    multiple="stack",
    bins=100,
    palette=["tab:blue", "tab:orange"],
    edgecolor="none",
    teeplot_subdir=teeplot_subdir,
) as ax:
    ax.set_xlabel("Layer")
    ax.set_ylabel("Mismatch Count")
    ax.set_title("Mismatches per Layer by Parity Error")

    ax2 = ax.twinx()
    frac_data = (
        df.groupby("layer")["parity_error"]
        .apply(lambda g: (g == "+1 parity").sum() / len(g) * 100)
        .reset_index(name="frac_plus1")
    )
    ax2.plot(
        frac_data["layer"],
        frac_data["frac_plus1"],
        color="black",
        linewidth=0.5,
        alpha=0.7,
        label="+1 parity %",
    )
    ax2.set_ylabel("+1 Parity %")
    ax2.set_ylim(0, 100)
    ax2.legend(loc="upper right", fontsize=5)

    sns.move_legend(ax, "upper left", fontsize=5)
    ax.figure.set_size_inches(1.5, 1.5)


## Plot 3 --- Column Distribution Along Row 410


In [ ]:
df_row410 = df[df["x"] == 410].copy()
print(f"{len(df_row410)} mismatches at row 410")

with tp.teed(
    sns.histplot,
    data=df_row410,
    x="y",
    hue="parity_error",
    multiple="stack",
    bins=50,
    palette=["tab:blue", "tab:orange"],
    edgecolor="none",
    teeplot_subdir=teeplot_subdir,
) as ax:
    ax.set_xlabel("Column")
    ax.set_ylabel("Mismatch Count")
    ax.set_title(f"Column Distribution at Row 410 (n={len(df_row410):,})")
    ax.set_xlim(0, nCol)
    sns.move_legend(ax, "upper right", fontsize=5)
    ax.figure.set_size_inches(1.5, 1.5)


In [ ]:
df_site_faults = df.groupby(["x", "y"]).size().reset_index(name="fault_count")
print(f"{len(df_site_faults)} unique sites with faults")
df_site_faults.head()


## Plot 4 --- Fault Count Heatmap


In [ ]:
grid = np.zeros((nRow, nCol), dtype=int)
grid[df_site_faults["x"].values, df_site_faults["y"].values] = df_site_faults[
    "fault_count"
].values
grid_masked = np.ma.masked_where(grid == 0, grid)

with tp.teed(
    sns.heatmap,
    data=grid_masked.filled(np.nan),
    cmap="YlOrRd",
    cbar_kws={"label": "Fault Count"},
    xticklabels=False,
    yticklabels=False,
    teeplot_subdir=teeplot_subdir,
) as ax:
    ax.set_xlabel("Column")
    ax.set_ylabel("Row")
    ax.set_title("Fault Count by Site (Heatmap)")
    ax.set_aspect("equal")
    ax.figure.set_size_inches(1.5, 1.5)


## Plot 5 --- Fault Count Scatterplot


In [ ]:
with tp.teed(
    sns.scatterplot,
    data=df_site_faults,
    x="y",
    y="x",
    hue="fault_count",
    size="fault_count",
    sizes=(1, 40),
    palette="YlOrRd",
    edgecolor="none",
    alpha=0.8,
    teeplot_subdir=teeplot_subdir,
) as ax:
    ax.set_xlim(0, nCol)
    ax.set_ylim(nRow, 0)
    ax.set_aspect("equal")
    ax.set_xlabel("Column")
    ax.set_ylabel("Row")
    ax.set_title("Fault Count by Site (Scatter)")
    sns.move_legend(
        ax,
        "upper right",
        fontsize=5,
        markerscale=0.5,
    )
    ax.figure.set_size_inches(1.5, 1.5)
